In [1]:
# nested dir structure
# multiple processes are writing/reading/deleting files

# given a filename, find files in the dir with that name, return the message, and delete the file

In [39]:
import dataclasses
import os
from typing import Final

In [5]:
@dataclasses.dataclass(kw_only=True, frozen=True)
class File:
    message: str

In [33]:
FILES = {
    "toplevel1": {
        "foo": File(message="here i am"),
        "bar": File(message=":)"),
    },
    "foo": File(message="hello world!"), 
}

In [40]:
DIRPATH: Final[str] = "sample_folder"

In [41]:
os.listdir(DIRPATH)

['foo.txt', 'toplevel1', '.ipynb_checkpoints']

In [63]:
def add_item(
    dir_prefix: str,
    name: str,
    curr_dir: dict,
) -> None:
    """
    Given an item, figure out whether it is a directory or a file.
    Add it recursively if it's a directory,
    else read in the file contents and add it with key `name`.
    """
    # ignore hidden files
    if name.startswith("."):
        return
    full_name = dir_prefix + "/" + name
    if os.path.isdir(full_name):
        curr_dir[name] = {}
        curr_dir = curr_dir[name]
        for item in os.listdir(full_name):
            add_item(
                dir_prefix=full_name,
                name=item,
                curr_dir=curr_dir,
            )
    else:
        with open(full_name) as f:
            curr_dir[name] = f.read()
    return

In [64]:
FILES = {}
curr_dir = FILES
prefix = DIRPATH
for item in os.listdir(DIRPATH):
    add_item(
        dir_prefix=prefix,
        name=item,
        curr_dir=curr_dir,
    )

In [65]:
FILES

{'foo.txt': 'hello, world!\n',
 'toplevel1': {'foo.txt': 'here i am\n', 'bar.txt': 'smiley\n'}}

In [30]:
def check_file_matches(
    dir_key: str,
    dir_value: dict | File,
    target_filename: str,
) -> bool:
    """Tell whether the current item is a file with the desired name."""
    is_file = isinstance(dir_value, File)
    has_target_name = dir_key == target_filename
    return is_file and has_target_name

In [36]:
def find_files(filename: str) -> list[str]:
    """
    Given a filename,
    find file(s) in the directory with that name,
    append the message to the output,
    and delete the file(s).

    If no matching files are found, return an empty list.
    """
    messages: list[str] = []
    subdirs: list[dict] = [FILES]
    while len(subdirs) > 0:
        curr_subdir = subdirs.pop()
        for key, value in curr_subdir.items():
            assert isinstance(key, str), f"Malformed key! {type(key)}: {key}"
            assert isinstance(value, (dict, File)), value
            if isinstance(value, dict):
                subdirs.append(value)
            if check_file_matches(dir_key=key, dir_value=value, target_filename=filename):
                messages.append(value.message)
                # TODO(sparsh): delete this from the directory
                # del curr_subdir[key]  # can't do this as it changes dict size during iteration -> error
                # os.remove(...)
    return messages

In [37]:
find_files("foo")

['hello world!', 'here i am']